In [ ]:
import os
import math
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy as sp
import scipy.signal as signal
from mpl_toolkits.axes_grid1 import make_axes_locatable

dtype = torch.float
# Check whether GPU is available
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

train_args = {
    'alpha': 1e-1,
    'beta': 1e0,
}

# ---------------------------------------------------------
# 1. Model Definitions: TCNN
# ---------------------------------------------------------

class TCNN(nn.Module):
    def __init__(self, TCNN_args):
        super(TCNN, self).__init__()
        self.Feature_detector = nn.Sequential(
            nn.Conv1d(in_channels=TCNN_args['in_channels'], out_channels=TCNN_args['out_channels'], kernel_size=TCNN_args['kernel_size']),
            nn.ReLU(),
        )
        self.Norm_Pool = nn.Sequential(
            nn.BatchNorm1d(num_features=TCNN_args['out_channels'], affine=False),
            nn.MaxPool1d(kernel_size=TCNN_args['maxpool_size']),
        )
        self.FC = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.Linear(in_features=TCNN_args['out_channels'] * int((TCNN_args['batch_size'] - TCNN_args['kernel_size'] + 1 - 2) / TCNN_args['maxpool_size'] + 1), out_features=TCNN_args['category_num'], bias=False),
        )

    def forward(self, x):
        return F.log_softmax(self.FC(x), dim=1)

def kernel_init_tcnn(run_TCNN, TCNN_args):
    for param in run_TCNN.Feature_detector.parameters():
        param.requires_grad = False
    run_TCNN_param = run_TCNN.state_dict()
    step = TCNN_args['kernel_size'] // 6 // TCNN_args['out_channels']
    for ii in range(0, TCNN_args['out_channels']):
        run_TCNN_param['Feature_detector.0.weight'][ii] = torch.tensor(signal.windows.gaussian(TCNN_args['kernel_size'], 2 + ii * step), dtype=dtype).to(device)
    run_TCNN_param['Feature_detector.0.bias'][:] = 0.0
    run_TCNN.load_state_dict(run_TCNN_param)
    return run_TCNN

# ---------------------------------------------------------
# 2. Model Definitions: TTE
# ---------------------------------------------------------

class PositionalEncoding(nn.Module):
    def __init__(self, num_hiddens, dropout, max_len=1000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(dropout)
        self.P = torch.zeros((1, max_len, num_hiddens))
        X = torch.arange(max_len, dtype=torch.float32).reshape(-1, 1) / torch.pow(10000, torch.arange(0, num_hiddens, 2, dtype=torch.float32) / num_hiddens)
        self.P[:, :, 0::2] = torch.sin(X)
        self.P[:, :, 1::2] = torch.cos(X[:,0:2])

    def forward(self, X):
        X = X + self.P[:, :X.shape[1], :].to(X.device)
        return self.dropout(X)

class ConvEmbeding(nn.Module):
    def __init__(self, TTE_args):
        super(ConvEmbeding, self).__init__()
        self.Feature_detector = nn.Sequential(
            nn.Conv1d(in_channels=TTE_args['input_channels'], out_channels=TTE_args['expected_dim'],
                    kernel_size=TTE_args['kernel_size'], stride=TTE_args['stride']),
            nn.ReLU(),
        )
        self.Norm_Pool = nn.Sequential(
            nn.BatchNorm1d(num_features=TTE_args['expected_dim'], affine=False),
            nn.MaxPool1d(kernel_size=TTE_args['maxpool_size']),
        )
        self.Embeding = nn.Sequential(
            self.Feature_detector,
            self.Norm_Pool,
        )

    def forward(self, x):
        return self.Embeding(x).permute(0,2,1)

class TemporalTransformerEncoder(nn.Module):
    def __init__(self, TTE_args):
        super(TemporalTransformerEncoder, self).__init__()
        self.Embeding = ConvEmbeding(TTE_args)
        self.PositionalEncoding = PositionalEncoding(TTE_args['expected_dim'], TTE_args['position_dropout'])
        self.transformer_encoder = nn.TransformerEncoder(nn.TransformerEncoderLayer(TTE_args['expected_dim'], TTE_args['num_heads'], batch_first=True), TTE_args['num_layers'])
        self.FC = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.LazyLinear(out_features=TTE_args['category_num']),
        )

    def forward(self, x):
        return F.log_softmax(self.FC(self.transformer_encoder(self.PositionalEncoding(x))), dim=1)

def kernel_init_tte(run_TTE, TTE_args):
    for param in run_TTE.Embeding.Feature_detector.parameters():
        param.requires_grad = False
    run_TTE_param = run_TTE.state_dict()
    step = TTE_args['kernel_size'] // 6 // TTE_args['expected_dim']
    for ii in range(0, TTE_args['expected_dim']):
        run_TTE_param['Embeding.Feature_detector.0.weight'][ii] = torch.tensor(signal.windows.gaussian(TTE_args['kernel_size'], 2 + ii * step), dtype=dtype).to(device)
    run_TTE_param['Embeding.Feature_detector.0.bias'][:] = 0.0
    run_TTE.load_state_dict(run_TTE_param)
    return run_TTE


# ---------------------------------------------------------
# 3. Utilities for Accuracy and Confusion Matrices
# ---------------------------------------------------------

def acu_cal(log_py_train, y_train, log_py_test, y_test, acu):
    # Overall Accuracy
    acu['acu_train'].append((torch.sum(torch.argmax(log_py_train, dim=1) == y_train) / y_train.size(0)).item())
    acu['acu_test'].append((torch.sum(torch.argmax(log_py_test, dim=1) == y_test) / y_test.size(0)).item())

    # Class 0: STDP
    if y_train[torch.argwhere(y_train == 0)].size(0) > 0:
        acu['acu_train_STDP'].append((torch.sum(torch.argmax(log_py_train, dim=1)[torch.argwhere(y_train == 0)] == y_train[torch.argwhere(y_train == 0)]) / y_train[torch.argwhere(y_train == 0)].size(0)).item())
    else:
        acu['acu_train_STDP'].append(0.0)

    if y_test[torch.argwhere(y_test == 0)].size(0) > 0:
        acu['acu_test_STDP'].append((torch.sum(torch.argmax(log_py_test, dim=1)[torch.argwhere(y_test == 0)] == y_test[torch.argwhere(y_test == 0)]) / y_test[torch.argwhere(y_test == 0)].size(0)).item())
    else:
        acu['acu_test_STDP'].append(0.0)

    # Class 1: BPTT1
    if y_train[torch.argwhere(y_train == 1)].size(0) > 0:
        acu['acu_train_BP1'].append((torch.sum(torch.argmax(log_py_train, dim=1)[torch.argwhere(y_train == 1)] == y_train[torch.argwhere(y_train == 1)]) / y_train[torch.argwhere(y_train == 1)].size(0)).item())
    else:
        acu['acu_train_BP1'].append(0.0)

    if y_test[torch.argwhere(y_test == 1)].size(0) > 0:
        acu['acu_test_BP1'].append((torch.sum(torch.argmax(log_py_test, dim=1)[torch.argwhere(y_test == 1)] == y_test[torch.argwhere(y_test == 1)]) / y_test[torch.argwhere(y_test == 1)].size(0)).item())
    else:
        acu['acu_test_BP1'].append(0.0)

    # Class 2: BPTT2
    if y_train[torch.argwhere(y_train == 2)].size(0) > 0:
        acu['acu_train_BP2'].append((torch.sum(torch.argmax(log_py_train, dim=1)[torch.argwhere(y_train == 2)] == y_train[torch.argwhere(y_train == 2)]) / y_train[torch.argwhere(y_train == 2)].size(0)).item())
    else:
        acu['acu_train_BP2'].append(0.0)

    if y_test[torch.argwhere(y_test == 2)].size(0) > 0:
        acu['acu_test_BP2'].append((torch.sum(torch.argmax(log_py_test, dim=1)[torch.argwhere(y_test == 2)] == y_test[torch.argwhere(y_test == 2)]) / y_test[torch.argwhere(y_test == 2)].size(0)).item())
    else:
        acu['acu_test_BP2'].append(0.0)

    # NOTE: NC class removed entirely.
    return acu

def confusion_cal(y_test, py_test, y_type):
    class_num = len(y_test.unique())
    confusion_num = np.zeros((class_num, class_num))
    if y_type == 0:
        for ii in range(0, class_num):
            for jj in range(0, class_num):
                confusion_num[ii,jj] = len(np.intersect1d(torch.argwhere(torch.argmax(py_test, dim=1) == jj).detach().cpu().numpy(), torch.argwhere(y_test == ii).detach().cpu().numpy()))
    elif y_type == 1:
        for ii in range(0, class_num):
            for jj in range(0, class_num):
                confusion_num[ii,jj] = len(np.intersect1d(torch.argwhere(py_test == jj).detach().cpu().numpy(), torch.argwhere(y_test == ii).detach().cpu().numpy()))
    return confusion_num

# ---------------------------------------------------------
# 4. Data Loading and Filtering
# ---------------------------------------------------------

dat_dir = '/content/drive/MyDrive/Project/PlasticityDecoding/data'

SPK = []
ptype = []
for rnum in range(0, 11):
    SPK.append(np.load(dat_dir + f'/EndtoEnd/Spk_an_wn_{rnum}.npy'))
    ptype.append(np.load(dat_dir + f'/EndtoEnd/Ptype_an_wn_{rnum}.npy'))

SPK = np.concatenate(SPK, axis=0)
ptype = np.concatenate(ptype, axis=0)

# Filter out NC class (3)
valid_idx = ptype != 3
SPK = SPK[valid_idx]
ptype = ptype[valid_idx]

SPK1 = []
ptype1 = []
for rnum in range(0, 32):
    SPK1.append(np.load(dat_dir + f'/EndtoEnd01/Spk_an_wn_{rnum}.npy'))
    ptype1.append(np.load(dat_dir + f'/EndtoEnd01/Ptype_an_wn_{rnum}.npy'))

SPK1 = np.concatenate(SPK1, axis=0)
ptype1 = np.concatenate(ptype1, axis=0)

# Filter out NC class (3)
valid_idx1 = ptype1 != 3
SPK1 = SPK1[valid_idx1]
ptype1 = ptype1[valid_idx1]


# ---------------------------------------------------------
# 5. Training Loop for both TCNN & TTE
# ---------------------------------------------------------

# Matrices changed from 10 to 8 columns (Removed Train NC / Test NC)
tcnn_acu_set = np.zeros((50, 8))
tcnn_acu_set_test = np.zeros((50, 8))

tte_acu_set = np.zeros((50, 8))
tte_acu_set_test = np.zeros((50, 8))

for ii in range(0, 50):
    print(f"Running iteration {ii + 1}/50...")

    # Train / Test Splitting
    train_ind = np.random.choice(np.size(SPK, axis=0), np.int32(0.8 * np.size(SPK, axis=0)), replace=False)
    test_ind = np.setdiff1d(np.arange(0, np.size(SPK, axis=0)), train_ind)
    test_ind_od = np.random.choice(np.size(SPK1, axis=0), np.int32(0.2 * np.size(SPK1, axis=0)), replace=False)

    x_train = torch.tensor(SPK[train_ind, :], dtype=dtype).to(device).unsqueeze(dim=1)
    y_train = torch.tensor(ptype[train_ind], dtype=torch.long).to(device)

    x_test = torch.tensor(SPK[test_ind, :], dtype=dtype).to(device).unsqueeze(dim=1)
    y_test = torch.tensor(ptype[test_ind], dtype=torch.long).to(device)

    x_test_od = torch.tensor(SPK1[test_ind_od, :], dtype=dtype).to(device).unsqueeze(dim=1)
    y_test_od = torch.tensor(ptype1[test_ind_od], dtype=torch.long).to(device)

    # ------------------
    # EVALUATING TCNN
    # ------------------

    TCNN_args = {
        'in_channels': 1,
        'out_channels': 5,
        'kernel_size': 500,
        'maxpool_size': 125,
        'category_num': 3, # Changed from 4 to 3
        'batch_size': x_train.size(dim=2),
    }

    acu_tcnn = {'acu_train': [], 'acu_test': [], 'acu_train_STDP': [], 'acu_test_STDP': [], 'acu_train_BP1': [], 'acu_test_BP1': [], 'acu_train_BP2': [], 'acu_test_BP2': []}
    acu_test_tcnn = {'acu_train': [], 'acu_test': [], 'acu_train_STDP': [], 'acu_test_STDP': [], 'acu_train_BP1': [], 'acu_test_BP1': [], 'acu_train_BP2': [], 'acu_test_BP2': []}

    run_TCNN = TCNN(TCNN_args).to(device)
    run_TCNN = kernel_init_tcnn(run_TCNN, TCNN_args)
    loss_fn = nn.NLLLoss()
    optimizer_tcnn = torch.optim.Adam(run_TCNN.parameters(), lr=1e-3)

    run_TCNN.train()
    x_train1_tcnn = run_TCNN.Norm_Pool(run_TCNN.Feature_detector(x_train))
    x_test1_tcnn = run_TCNN.Norm_Pool(run_TCNN.Feature_detector(x_test))
    x_test_od1_tcnn = run_TCNN.Norm_Pool(run_TCNN.Feature_detector(x_test_od))

    for epoch in range(0, 500):
        log_py_train = run_TCNN(x_train1_tcnn)
        for weight in run_TCNN.FC.parameters():
            L1 = train_args['beta'] * F.l1_loss(torch.zeros_like(weight), weight)
        L = train_args['alpha'] * loss_fn(log_py_train, y_train) + L1
        optimizer_tcnn.zero_grad()
        L.backward()
        optimizer_tcnn.step()

        if torch.sum(torch.argmax(log_py_train, dim=1) == y_train) / y_train.size(0) > 0.85:
            break

    run_TCNN.eval()
    log_py_test_tcnn = run_TCNN(x_test1_tcnn)
    acu_tcnn = acu_cal(log_py_train, y_train, log_py_test_tcnn, y_test, acu_tcnn)

    log_py_test_od_tcnn = run_TCNN(x_test_od1_tcnn)
    acu_test_tcnn = acu_cal(log_py_train, y_train, log_py_test_od_tcnn, y_test_od, acu_test_tcnn)

    # Save Iteration metrics for TCNN
    tcnn_acu_set[ii, 0], tcnn_acu_set[ii, 1] = acu_tcnn['acu_train'][0], acu_tcnn['acu_test'][0]
    tcnn_acu_set[ii, 2], tcnn_acu_set[ii, 3] = acu_tcnn['acu_train_STDP'][0], acu_tcnn['acu_test_STDP'][0]
    tcnn_acu_set[ii, 4], tcnn_acu_set[ii, 5] = acu_tcnn['acu_train_BP1'][0], acu_tcnn['acu_test_BP1'][0]
    tcnn_acu_set[ii, 6], tcnn_acu_set[ii, 7] = acu_tcnn['acu_train_BP2'][0], acu_tcnn['acu_test_BP2'][0]

    tcnn_acu_set_test[ii, 0], tcnn_acu_set_test[ii, 1] = acu_test_tcnn['acu_train'][0], acu_test_tcnn['acu_test'][0]
    tcnn_acu_set_test[ii, 2], tcnn_acu_set_test[ii, 3] = acu_test_tcnn['acu_train_STDP'][0], acu_test_tcnn['acu_test_STDP'][0]
    tcnn_acu_set_test[ii, 4], tcnn_acu_set_test[ii, 5] = acu_test_tcnn['acu_train_BP1'][0], acu_test_tcnn['acu_test_BP1'][0]
    tcnn_acu_set_test[ii, 6], tcnn_acu_set_test[ii, 7] = acu_test_tcnn['acu_train_BP2'][0], acu_test_tcnn['acu_test_BP2'][0]


    # ------------------
    # EVALUATING TTE
    # ------------------

    TTE_args = {
        'input_channels': 1,
        'kernel_size': 500,
        'stride': 1,
        'maxpool_size': 125,
        'num_heads': 5,
        'expected_dim': 5,
        'num_layers': 1,
        'category_num': 3, # Changed from 4 to 3
        'position_dropout': 0.0,
        'data_len': x_train.size(dim=1),
    }

    acu_tte = {'acu_train': [], 'acu_test': [], 'acu_train_STDP': [], 'acu_test_STDP': [], 'acu_train_BP1': [], 'acu_test_BP1': [], 'acu_train_BP2': [], 'acu_test_BP2': []}
    acu_test_tte = {'acu_train': [], 'acu_test': [], 'acu_train_STDP': [], 'acu_test_STDP': [], 'acu_train_BP1': [], 'acu_test_BP1': [], 'acu_train_BP2': [], 'acu_test_BP2': []}

    run_TTE = TemporalTransformerEncoder(TTE_args).to(device)
    run_TTE = kernel_init_tte(run_TTE, TTE_args)
    optimizer_tte = torch.optim.Adam(run_TTE.parameters(), lr=1e-3)

    run_TTE.train()
    x_train1_tte = run_TTE.Embeding(x_train)
    x_test1_tte = run_TTE.Embeding(x_test)
    x_test_od1_tte = run_TTE.Embeding(x_test_od)

    for epoch in range(0, 500):
        log_py_train = run_TTE(x_train1_tte)
        L = loss_fn(log_py_train, y_train)
        optimizer_tte.zero_grad()
        L.backward()
        optimizer_tte.step()

        if torch.sum(torch.argmax(log_py_train, dim=1) == y_train) / y_train.size(0) > 0.85:
            break

    run_TTE.eval()
    log_py_test_tte = run_TTE(x_test1_tte)
    acu_tte = acu_cal(log_py_train, y_train, log_py_test_tte, y_test, acu_tte)

    log_py_test_od_tte = run_TTE(x_test_od1_tte)
    acu_test_tte = acu_cal(log_py_train, y_train, log_py_test_od_tte, y_test_od, acu_test_tte)

    # Save Iteration metrics for TTE
    tte_acu_set[ii, 0], tte_acu_set[ii, 1] = acu_tte['acu_train'][0], acu_tte['acu_test'][0]
    tte_acu_set[ii, 2], tte_acu_set[ii, 3] = acu_tte['acu_train_STDP'][0], acu_tte['acu_test_STDP'][0]
    tte_acu_set[ii, 4], tte_acu_set[ii, 5] = acu_tte['acu_train_BP1'][0], acu_tte['acu_test_BP1'][0]
    tte_acu_set[ii, 6], tte_acu_set[ii, 7] = acu_tte['acu_train_BP2'][0], acu_tte['acu_test_BP2'][0]

    tte_acu_set_test[ii, 0], tte_acu_set_test[ii, 1] = acu_test_tte['acu_train'][0], acu_test_tte['acu_test'][0]
    tte_acu_set_test[ii, 2], tte_acu_set_test[ii, 3] = acu_test_tte['acu_train_STDP'][0], acu_test_tte['acu_test_STDP'][0]
    tte_acu_set_test[ii, 4], tte_acu_set_test[ii, 5] = acu_test_tte['acu_train_BP1'][0], acu_test_tte['acu_test_BP1'][0]
    tte_acu_set_test[ii, 6], tte_acu_set_test[ii, 7] = acu_test_tte['acu_train_BP2'][0], acu_test_tte['acu_test_BP2'][0]


# ---------------------------------------------------------
# 6. Save Data to Disk
# ---------------------------------------------------------

save_dir = '/content/drive/MyDrive/Project/PlasticityDecoding/result_20260601'
os.makedirs(save_dir, exist_ok=True)

# TCNN Saves
np.save(save_dir + '/TCNN_endtoend_no_NC.npy', tcnn_acu_set)
np.save(save_dir + '/TCNN_endtoend_no_NC_out_dist.npy', tcnn_acu_set_test)
np.save(save_dir + '/TCNN_endtoend_no_NC_confusion.npy', confusion_cal(y_test, log_py_test_tcnn, y_type=0))
np.save(save_dir + '/TCNN_endtoend_no_NC_confusion_out_dist.npy', confusion_cal(y_test_od, log_py_test_od_tcnn, y_type=0))

# TTE Saves
np.save(save_dir + '/TTE_endtoend_no_NC.npy', tte_acu_set)
np.save(save_dir + '/TTE_endtoend_no_NC_out_dist.npy', tte_acu_set_test)
np.save(save_dir + '/TTE_endtoend_no_NC_confusion.npy', confusion_cal(y_test, log_py_test_tte, y_type=0))
np.save(save_dir + '/TTE_endtoend_no_NC_confusion_out_dist.npy', confusion_cal(y_test_od, log_py_test_od_tte, y_type=0))

tcnn_in = np.round(tcnn_acu_set.mean(axis=0), 3)
tcnn_out = np.round(tcnn_acu_set_test.mean(axis=0), 3)

print("==============================")
print("Results: TCNN without NC class")
print("==============================")
print("[In-Distribution Accuracies (Train / Test)]")
print(f"Overall : {tcnn_in[0]:.3f} / {tcnn_in[1]:.3f}")
print(f"STDP    : {tcnn_in[2]:.3f} / {tcnn_in[3]:.3f}")
print(f"BPTT1   : {tcnn_in[4]:.3f} / {tcnn_in[5]:.3f}")
print(f"BPTT2   : {tcnn_in[6]:.3f} / {tcnn_in[7]:.3f}")
print("\n[Out-Distribution Accuracies (Train / Test)]")
print(f"Overall : {tcnn_out[0]:.3f} / {tcnn_out[1]:.3f}")
print(f"STDP    : {tcnn_out[2]:.3f} / {tcnn_out[3]:.3f}")
print(f"BPTT1   : {tcnn_out[4]:.3f} / {tcnn_out[5]:.3f}")
print(f"BPTT2   : {tcnn_out[6]:.3f} / {tcnn_out[7]:.3f}\n")

tte_in = np.round(tte_acu_set.mean(axis=0), 3)
tte_out = np.round(tte_acu_set_test.mean(axis=0), 3)

print("==============================")
print("Results: TTE without NC class")
print("==============================")
print("[In-Distribution Accuracies (Train / Test)]")
print(f"Overall : {tte_in[0]:.3f} / {tte_in[1]:.3f}")
print(f"STDP    : {tte_in[2]:.3f} / {tte_in[3]:.3f}")
print(f"BPTT1   : {tte_in[4]:.3f} / {tte_in[5]:.3f}")
print(f"BPTT2   : {tte_in[6]:.3f} / {tte_in[7]:.3f}")
print("\n[Out-Distribution Accuracies (Train / Test)]")
print(f"Overall : {tte_out[0]:.3f} / {tte_out[1]:.3f}")
print(f"STDP    : {tte_out[2]:.3f} / {tte_out[3]:.3f}")
print(f"BPTT1   : {tte_out[4]:.3f} / {tte_out[5]:.3f}")
print(f"BPTT2   : {tte_out[6]:.3f} / {tte_out[7]:.3f}\n")

print("Done! Check your GDrive folder for the resulting matrix exports.")

Running iteration 1/50...


/tmp/ipykernel_1330/1719045874.py:102: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.transformer_encoder = nn.TransformerEncoder(nn.TransformerEncoderLayer(TTE_args['expected_dim'], TTE_args['num_heads'], batch_first=True), TTE_args['num_layers'])


Running iteration 2/50...
Running iteration 3/50...
Running iteration 4/50...
Running iteration 5/50...
Running iteration 6/50...
Running iteration 7/50...
Running iteration 8/50...
Running iteration 9/50...
Running iteration 10/50...
Running iteration 11/50...
Running iteration 12/50...
Running iteration 13/50...
Running iteration 14/50...
Running iteration 15/50...
Running iteration 16/50...
Running iteration 17/50...
Running iteration 18/50...
Running iteration 19/50...
Running iteration 20/50...
Running iteration 21/50...
Running iteration 22/50...
Running iteration 23/50...
Running iteration 24/50...
Running iteration 25/50...
Running iteration 26/50...
Running iteration 27/50...
Running iteration 28/50...
Running iteration 29/50...
Running iteration 30/50...
Running iteration 31/50...
Running iteration 32/50...
Running iteration 33/50...
Running iteration 34/50...
Running iteration 35/50...
Running iteration 36/50...
Running iteration 37/50...
Running iteration 38/50...
Running i